In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# True states

In [180]:
# Time setup
T = 30
N = 600 # 800
dt = T / N
times = np.linspace(0,T,num=N)
num_steps = len(times)

# SDE: mean-reverting spiral
mus = np.array([1.5, 0.5]) # mu_sigma, mu_omega
thetas = np.array([0.1, 0.05]) # theta_sigma, theta_omega
sigmas = np.array([0.1, 0.05]) # sigma_sigma, sigma_omega

X = np.zeros((num_steps,5))
X[0,:] = [0, 0, mus[0], 0.2, mus[1]]
diffusion = np.zeros((5,2))

diffusion[2,0] = sigmas[0]
diffusion[4,1] = sigmas[1]

# Euler-Maryuma method
for i in range(1,num_steps):
  # previous states
  X_minus = X[i-1,:] # x1 x2 speed direction ang.velocity
  # drift
  drift = np.array([
      X_minus[2]*math.cos(X_minus[3]), # dx1
      X_minus[2]*math.sin(X_minus[3]), # dx2
      thetas[0]*(mus[0] - X_minus[2]), # ds
      X_minus[4], # dphi
      thetas[1]*(mus[1] - X_minus[4]) # dx3
  ])
  z = np.array([
      np.sqrt(dt)*np.random.normal(0,1),
      np.sqrt(dt)*np.random.normal(0,1)
  ])
  X[i,:] = X_minus + drift*dt + (diffusion @ z).T
  # angle normalzation
  X[i,3] = np.arctan2(np.sin(X[i,3]),np.cos(X[i,3]))

# Sensors and observations

In [181]:
# The sensors
radius = 12
sensors_xy = np.array([
    [0, radius],
    [radius * np.cos(7*np.pi/6), radius * np.sin(7*np.pi/6)],
    [radius * np.cos(11*np.pi/6), radius * np.sin(11*np.pi/6)]
])
# The observations
sigma_r = 0.15 # range noise
data_points = X[:,0:2]
true_y = cdist(data_points, sensors_xy)
noisy_y = true_y + np.random.normal(0,sigma_r,true_y.shape)
# noise covariances
Rhat = np.diag([sigma_r**2, sigma_r**2, sigma_r**2])
Qhat = np.diag([
    0.0,
    0.0,
    sigmas[0]**2 *dt,
    0.0,
    sigmas[1]**2 *dt
])


# Adaptive noise parameters

In [182]:
def F_matrix(X, thetas, dt):
    F = np.eye(5)
    v   = X[2]
    psi = X[3]

    F[0, 2] = np.cos(psi) * dt  # d(px)/dv
    F[0, 3] = -v * np.sin(psi) * dt # d(px)/dpsi
    F[1, 2] = np.sin(psi) * dt  # d(py)/dv
    F[1, 3] =  v * np.cos(psi) * dt # d(py)/dpsi
    F[2, 2] = np.exp(-thetas[0] * dt) # d(v)/dv
    F[3, 4] = dt  # d(psi)/domega
    F[4, 4] = np.exp(-thetas[1] * dt) # d(omega)/domega

    return F

# EKF Predict

In [183]:
# EKF update step
def ekf_predict(X_input, P_input, Q, thetas, mus, dt):
  # m⁻_k = f(m_{k-1})
  pos_x = X_input[0] # x
  pos_y = X_input[1] # y
  velo = X_input[2] # velocity
  psi = X_input[3] # direction
  ang_velo = X_input[4] # angular velocity
  # OU mean-reversion coefficients
  av = np.exp(-thetas[0]*dt)
  aw = np.exp(-thetas[1]*dt)
  # mean-reverting (deterministic parts)
  velo_next = mus[0] + (velo - mus[0])*av
  ang_velo_next = mus[1] + (ang_velo - mus[1])*aw
  psi_next = psi + ang_velo*dt
  # position update
  pos_x_next = pos_x + velo * np.cos(psi) * dt
  pos_y_next = pos_y + velo * np.sin(psi) * dt
  # store prediction
  X_pred = np.array([pos_x_next, pos_y_next, velo_next, psi_next, ang_velo_next])
  # get Jacobian F
  F = F_matrix(X_input,thetas, dt)
  # covariance prediction
  P_pred = F @ P_input @ F.T + Q
  P_pred = 0.5*(P_pred + P_pred.T) # enforce symmetry

  return [X_pred, P_pred, F]

# EKF Update

In [185]:
def ekf_update(X_in, P_in, z, sensors_xy, R, Q, F_prev, P_prev, k, w):
  pos_x = X_in[0] # x
  pos_y = X_in[1] # y

  m  = z.size # number of measurements
  n = X_in.size # number of states
  # h and H
  h = np.zeros(m)
  H = np.zeros((m,n))
  # create the matrices
  for i in range(m):
    xi = sensors_xy[i,0]
    yi = sensors_xy[i,1]

    distx = pos_x - xi
    disty = pos_y - yi
    r = np.sqrt(distx**2 + disty**2) + 1e-9
    h[i] = r
    # dh/dph and dh/dpy
    H[i,0] = distx/r
    H[i,1] = disty/r
  # innovation term
  v = z - h
  # adaptive noise parameters
  min_var=1e-12
  alpha = 0.95
  rmin = 1e-6
  rmax = 1e6
  qmin = 1e-8
  qmax = 1e4
  # innovation covariance
  S = H @ P_in @ H.T + R
  S = 0.5*(S + S.T) + 1e-9*np.eye(m) # symmetry and jitter
  # kalman gain
  K = P_in @ H.T @ np.linalg.solve(S, np.eye(m))
  # state update
  x_upd = X_in + K @ v
  # covariance update
  P_upd = (np.eye(n) - K @ H) @ P_in @ (np.eye(n) - K @ H).T + K @ R @ K.T
  P_upd = 0.5*(P_upd + P_upd.T)
  # adaptive R
  h_upd = np.zeros(m)
  for i in range(m):
    xi = sensors_xy[i,0]
    yi = sensors_xy[i,1]
    dx = x_upd[0] - xi
    dy = x_upd[1] - yi
    h_upd[i] = np.sqrt(dx**2 + dy**2)
  eps = z - h_upd
  R_tmp = alpha*R + (1.0 - alpha)*(np.outer(eps,eps) + H @ P_in @ H.T )
  R_new_diag = np.clip(np.diag(R_tmp), rmin, rmax)
  R_new_diag2 = np.maximum(R_new_diag,min_var)
  R_new = np.diag(R_new_diag2) # diagonality
  e = x_upd - X_in
  # adaptive Q
  Q_tmp = alpha*Q + (1.0 - alpha)*(K @ np.outer(v,v) @ K.T)
  Q_new_diag = np.clip(np.diag(Q_tmp), qmin, qmax)
  Q_new_diag = np.maximum(Q_new_diag,min_var)
  Q_new = np.diag(Q_new_diag) # diagonality

  return [x_upd, P_upd, R_new, Q_new]

# Extended Kalman Filter run

In [186]:
# EKF
X_est = np.zeros((num_steps,5))
P = np.zeros((5,5, num_steps))
# initial guesses
X_est[0,:] = X[0,:]
P[:,:,0] = np.diag([1, 1, 1, 0.5, 0.5])
# window_size for adaptive R and Q
window_size = 50 # 50
# run EKF
for k in range(1,num_steps):
  # 1) PREDICTION STEP
  x_current = X_est[k-1,:]
  P_current = P[:,:,k-1]
  [X_pred, P_pred, F] = ekf_predict(x_current, P_current, Qhat, thetas, mus, dt)
  # 3) UPDATE STEP
  [X_pred, P_pred, Rhat, Qhat] = ekf_update(
        X_pred, P_pred, noisy_y[k,:],
        sensors_xy, Rhat, Qhat,
        F, P_current, k,
        window_size)
  X_pred[3] = np.arctan2(np.sin(X_pred[3]),np.cos(X_pred[3]))
  X_est[k,:] = X_pred
  P[:,:,k] = P_pred

# Plotting

In [194]:
# x1 error
diff2 = X_est[:, 0] - data_points[:, 0]
pos_error2 = np.sqrt(np.sum(diff2**2))
rmse_pos2 = np.sqrt(np.mean(pos_error2**2))
print(f"Position RMSE (x1): {rmse_pos2}")
# x2
diff3 = X_est[:, 1] - data_points[:, 1]
pos_error3 = np.sqrt(np.sum(diff3**2))
rmse_pos3 = np.sqrt(np.mean(pos_error3**2))
print(f"Position RMSE (x2): {rmse_pos2}")
plt.figure(figsize=(6,4))
plt.plot(data_points[:,0],label="True x") # true values
plt.plot(X_est[:,0], label="Estimated x") # estimated values
plt.xlabel('time')
plt.ylabel('x')
plt.axis('equal')
plt.legend()
plt.title("State x_1 errors")
plt.savefig("x1_error.png", dpi=300)
plt.close()

Position RMSE (x1): 1.6508026125569972
Position RMSE (x2): 1.6508026125569972


In [195]:
plt.figure(figsize=(6,4))
plt.plot(data_points[:,1],label="True x") # true values
plt.plot(X_est[:,1], label="Estimated x") # estimated values
plt.xlabel('time')
plt.ylabel('x')
plt.axis('equal')
plt.legend()
plt.title("State x_2 errors")
plt.savefig("x2_error.png", dpi=300)
plt.close()

# Position

In [196]:
diff = X_est[:, 0:2] - data_points[:, 0:2]
pos_error = np.sqrt(np.sum(diff**2, axis=1))
rmse_pos = np.sqrt(np.mean(pos_error**2))
print(f"Position RMSE: {rmse_pos}")
# plotting (x,y)
plt.figure(figsize=(6,4))
plt.plot(data_points[:,0],data_points[:,1]) # true values
plt.plot(X_est[:,0],X_est[:,1]) # estimated values
plt.plot(sensors_xy[:,0],sensors_xy[:,1],'ro') # sensors
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.title("Trajectory")
plt.savefig("trajectory.png", dpi=300)
plt.close()

Position RMSE: 0.08860319408882629


# Velocity

In [197]:
velo_error = X_est[:,2] - X[:,2]
rmse_velo = np.sqrt(np.mean(velo_error**2))
print(f"Velocity RMSE: {rmse_velo}")
# plotting
plt.figure(figsize=(10,10))
plt.plot(times,X_est[:,2], 'r--') # true values
plt.plot(times,X[:,2], 'b--') # estimated values
plt.xlabel('Time(s)')
plt.ylabel('Velocity (m/s)')
plt.axis('equal')
plt.title("Speed")
plt.savefig("speed.png", dpi=300)
plt.close()

Velocity RMSE: 0.202283029073725


# Heading

In [198]:
heading_diff = np.arctan2(
    np.sin(X_est[:,3] - X[:,3]),
    np.cos(X_est[:,3] - X[:,3])
)

# RMSE in radians
rmse_heading_rad = np.sqrt(np.mean(heading_diff**2))
# convert to degrees only for reporting
rmse_heading_deg = np.degrees(rmse_heading_rad)

print(f"Corrected Heading RMSE: {rmse_heading_deg:.3f} degrees")
# plotting
plt.figure(figsize=(10,10))
plt.plot(times,X_est[:,3], 'r--') # true values
plt.plot(times,X[:,3], 'b--') # estimated values
plt.xlabel('Time(s)')
plt.ylabel('rad')
plt.axis('equal')
plt.title("Heading")
plt.savefig("heading.png", dpi=300)
plt.close()

Corrected Heading RMSE: 4.983 degrees


# Turn rate

In [199]:
omega_error = X_est[:,4] - X[:,4]
rmse_omega_rad = np.sqrt(np.mean(omega_error**2))

print(f"Turn rate RMSE: {rmse_omega_rad:.6f} rad/s")
# plotting
plt.figure(figsize=(10,10))
plt.plot(times,X_est[:,4], 'r--') # true values
plt.plot(times,X[:,4],'b--') # estimated values
plt.xlabel('Time(s)')
plt.ylabel('rad')
plt.axis('equal')
plt.title("Turn rate")
plt.savefig("turn.png", dpi=300)
plt.close()

Turn rate RMSE: 0.059795 rad/s
